**Table of contents**<a id='toc0_'></a>    
- 1. [Using plumber](#toc1_)    
  - 1.1. [Get All pdf files tables](#toc1_1_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[Using plumber](#toc0_)

In [1]:
import os
from pprint import pprint

import numpy as np
import pandas as pd
import pdfplumber
from tqdm.notebook import tqdm

## 1.1. <a id='toc1_1_'></a>[Get All pdf files tables](#toc0_)

In [2]:
# get all pdf file in a directory that contain keywords in there names
def get_pdfs_with_keyword(directory, keywords:list[str]):
    pdf_files_list = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith(".pdf"):
                if any(keyword in file for keyword in keywords):
                    pdf_files_list.append(os.path.join(root, file))
    return pdf_files_list

In [6]:
def get_potable_water_data(potable_pdf_reprts):
    assert len(potable_pdf_reprts) > 0, "No pdf files Found!"
    print(f"{len(potable_pdf_reprts)} pdf files found")
    all_df = pd.DataFrame()

    for pdf_file in tqdm(potable_pdf_reprts):
        with pdfplumber.open(pdf_file) as pdf:
            tables = [page.extract_table() for page in pdf.pages[:]]
            print(len(tables))
            for table in tables[:]:
                df = pd.DataFrame(table)
                if (pdf_file[-8:-6] == "20"):
                    df["date"] = "".join([pdf_file[-14:-8] + pdf_file[-6:-4]])
                    # print("Woooow:","".join([pdf_file[-14:-8] + pdf_file[-6:-4]]))
                else:
                    df["date"] = pdf_file[-12:-4]
                    # print(pdf_file[-12:-4])
                all_df = pd.concat([all_df, df], axis=0)
    return all_df

In [7]:
# Replace 'your_directory_path' with the path to the directory you want to search
directory_path = r'../LAB2A'
keywords =['RADEES', 'RADESS']
potable_pdf_reprts = get_pdfs_with_keyword(directory_path, keywords)

pprint(potable_pdf_reprts)

['../LAB2A\\Rapport potablité Eau RADESS 01-05-2027.pdf',
 '../LAB2A\\Rapport potablité Eau RADESS 02-05-2027.pdf']


In [34]:
import camelot
import PyPDF2
import tempfile
import os

URL_FILE = potable_pdf_reprts[0]


# @st.cache_data
def get_table_from_pdf(uploaded_file: str = URL_FILE, method="lattice"):
    if True:  # path_file.endswith(".pdf"):
        try:
            # path_file=r"files/DRI_Micronutriments.pdf"
            tables = camelot.read_pdf(filepath=uploaded_file, pages="1-end", flavor=method)
            print(f"{uploaded_file} successfully loaded!")
            if len(tables) > 0:
                print(f"{len(tables)} tables extracted")
                return tables
            else:
                raise Exception("No table extracted!")
        except Exception as e:
            print(e)
            print("Try change -method- parameter to 'stream'!")
            # tables = camelot.read_pdf(filepath=url_file, pages="all", flavor="lattice")
    else:
        raise Exception("File is not a PDF")


def get_tables_info(tables) -> pd.DataFrame:
    assert len(tables) > 0, "No tables Found!"
    table_infos = [
        table.parsing_report | {"n_rows": table.shape[0], "n_cols": table.shape[1]}
        for table in tables
    ]
    return pd.DataFrame(table_infos)


def get_pdf_file_info(pdf_file):
    if pdf_file is not None:
        pdf_reader = PyPDF2.PdfFileReader(pdf_file)
        num_pages = pdf_reader.getNumPages()
        print(f"Number of pages: {num_pages}")
        # Get the full path of the PDF file
        pdf_path = pdf_reader.stream.name
        print("Name : ", pdf_path)
        print("Info : ", pdf_reader.getDocumentInfo())


print("hi")
tables = get_table_from_pdf(method="stream")
print(get_tables_info(tables))


hi
../LAB2A\Rapport potablité Eau RADESS 01-05-2027.pdf successfully loaded!
2 tables extracted
   accuracy  whitespace  order  page  n_rows  n_cols
0     96.54       31.11      1     1       9       5
1     95.22       46.55      2     1      29      10


In [115]:
table1: pd.DataFrame = tables[1].df.iloc[3:,0:4]
table1 = table1.rename({0:'parametre', 1:'unite', 2:'critere', 3:'result'}, axis=1).replace({"":np.nan}).dropna()
table2 = tables[1].df.iloc[3:15,5:9]
table2 = table2.rename({5:'parametre', 6:'unite', 7:'critere', 8:'result'}, axis=1).replace({"":np.nan}).dropna()
pd.concat([table1, table2]).reset_index(drop=True)

,parametre,unite,critere,result
0,pH à 20°C,U.pH,"6,5 - 8,5","6,8"
1,CE à 20°C,µS/cm,< 2700,474
2,Odeur*,IO,**,<1
3,Saveur*,IO,**,0
4,Température (EAU)*,°C,**,"21,30"
5,Ammonium,mg/L,"< 0,5",<0.016
6,Arsenic*,mg/L,**,<0.01
7,Baryum,mg/L,**,<0.01
8,Cadmium,mg/L,**,<0.01
9,Plomb*,mg/L,**,<0.01


In [8]:
water_dataframe: pd.DataFrame = get_potable_water_data(potable_pdf_reprts[:])
water_dataframe

2 pdf files found


  0%|          | 0/2 [00:00<?, ?it/s]

1
1


,0,1,2,3,4,date
0,Paramètres,Unité,Critères,Résultats,Niveau,01-05-27
1,pH à 20°C,U.pH,"6,5 - 8,5","6,8",,01-05-27
2,CE à 20°C,µS/cm,< 2700,474,,01-05-27
3,Odeur*,IO,**,<1,,01-05-27
4,Saveur*,IO,**,0,,01-05-27
5,Température (EAU)*,°C,**,"21,30",,01-05-27
6,,Unité,Critères,Résultats,Niveau,01-05-27
7,Ammonium,mg/L,"< 0,5",<0.016,,01-05-27
8,Arsenic*,mg/L,**,<0.01,,01-05-27
9,Baryum,mg/L,**,<0.01,,01-05-27


In [6]:
# For a pandas DataFrame
def find_result_in_dataframe(df, value="Résultat"):
    result = df.eq(value)
    if result.any().any():
        index, column = np.where(result)
        return index[0], df.columns[column[0]]
    return None, None
# get index and columns for the first value "Result"
i,c = find_result_in_dataframe(water_dataframe, value="Résultat")
print(i,c)
# drop the first 7 lignes from the dataframe
df: pd.DataFrame = water_dataframe.drop(index=range(i))
df.iloc[0, 0] = "Paramètre"
df.iloc[0, c] = "Résul"
df = df.set_axis(df.iloc[0], axis="columns")

date_col = [col for col in df.columns.dropna() if col.endswith(("23","24", "25", "26", "RADEES"))][0]
df.rename(columns={date_col: "date"}, inplace=True)
df = df[[col for col in df.columns.dropna() if col.startswith(("Param", "Résu", "date"))]]
df.insert(0, "date", df.pop('date'))
try:
    df.drop(columns=['Paramètre(s) microbiologiques'], inplace=True)
except Exception:
    pass
df = df.dropna(subset="Paramètre")
# df.sort_values(by='date', inplace=True)

df.query("Paramètre != ''")
df["Résultat"] = df.pop("Résul")
df = df.dropna(subset=["Résultat"]).query("(Résultat != 'Résultat') and (Résultat != 'Résul')")
df

None None


TypeError: 'NoneType' object cannot be interpreted as an integer

In [34]:
print(df.isna().sum().sum())
assert df.date.nunique() == len(potable_pdf_reprts), f"some files are skiped:{df.date.nunique()} # {len(potable_pdf_reprts)}"


0


In [35]:
df.to_excel(r'../outputs/data.xlsx', index=False)

---

In [ ]:
import camelot
import streamlit as st
import pandas as pd
import PyPDF2
import tempfile
import os

URL_FILE = "https://camelot-py.readthedocs.io/en/master/_static/pdf/foo.pdf"

# @st.cache_data
def get_table_from_pdf(
    uploaded_file: str = URL_FILE, method="lattice"):
    with tempfile.NamedTemporaryFile(delete=False, suffix='.pdf') as tmp_file:
        tmp_file.write(uploaded_file.read())
        path_file = tmp_file.name
        st.success(f"File saved at:{path_file}")
        # Get the tables in the PDF file
        if True:#path_file.endswith(".pdf"):
            try:
                # path_file=r"files/DRI_Micronutriments.pdf"
                tables = camelot.read_pdf(filepath=path_file, pages="1-end", flavor=method)
                st.success(f"{path_file} successfully loaded!")
                if len(tables) > 0:
                    print(f"{len(tables)} tables extracted")
                    return tables
                else:
                    raise Exception("No table extracted!")
            except Exception as e:
                st.exception(e)
                print("Try change -method- parameter to 'stream'!")
                # tables = camelot.read_pdf(filepath=url_file, pages="all", flavor="lattice")
        else:
            raise Exception("File is not a PDF")


def get_tables_info(tables) -> pd.DataFrame:
    assert len(tables) > 0, "No tables Found!"
    table_infos = [
        table.parsing_report | {"n_rows": table.shape[0], "n_cols": table.shape[1]}
        for table in tables
    ]
    return pd.DataFrame(table_infos)


def get_pdf_file_info(pdf_file):
    if pdf_file is not None:
        pdf_reader = PyPDF2.PdfFileReader(pdf_file)
        num_pages = pdf_reader.getNumPages()
        st.write(f"Number of pages: {num_pages}")
        # Get the full path of the PDF file
        pdf_path = pdf_reader.stream.name
        st.write("Name : ", pdf_path)
        st.write("Info : ", pdf_reader.getDocumentInfo())
        


if __name__ == '__main__':
    print('hi')
    tables = get_table_from_pdf(method='stream')
    print(get_tables_info(tables))
    
